# Day 32 · 多模态 RAG

**配套讲义**: [`days/day-32.md`](../days/day-32.md) ｜ **需要 GPU（云机器）**

搭商品图文双索引：**CLIP 向量做图搜**（用户上传图 → 找到对应商品）+ 文本向量做知识检索（退货政策、尺码表），并实现混合检索 + VLM 重排。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w6.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 跑自检，看「以图搜图」到底行不行

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.agent.retriever", "--demo"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout[-2500:] or r.stderr[-2500:])

## 2. 归一化漏了的后果（亲手验证）

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
q  = rng.normal(size=8)
a  = rng.normal(size=8) * 0.2      # 同方向但模长很小
b  = rng.normal(size=8) * 5.0      # 另一个方向但模长很大

def cos(x, y):  return float(x @ y / (np.linalg.norm(x) * np.linalg.norm(y)))
def dot(x, y):  return float(x @ y)

print("a 与 q 的真实相似度(cos):", round(cos(q, a), 4))
print("b 与 q 的真实相似度(cos):", round(cos(q, b), 4))
print()
print("不做归一化时：a 的内积", round(dot(q, a), 3), " b 的内积", round(dot(q, b), 3))
print("→ 如果不归一化，模长大的向量会霸榜，余弦相似度排序被彻底破坏")

## 3. 设计你的 RAG 评测集

**不测准确率就等于没做**。写 10 条「传图 → 期望商品 ID」。

In [ ]:
retrieval_eval = [
    # ("图片路径", "期望的商品ID"),
    # ("data/raw_images/tee_white_01.jpg", "SKU-1001"),
]
print(f"评测集 {len(retrieval_eval)} 条 —— 至少写 10 条，跑一遍算 top-3 命中率")

## 验收清单

- [ ] 图搜 top-3 准确率 ≥ 70%（在 100 张商品图上测，要人工核对）
- [ ] 混合检索能同时吃到图片信号和文本信号
- [ ] 能解释「以图搜图」比「先描述再搜文本」好在哪（**信息损失**角度）
- [ ] 知道归一化漏了会有什么后果（相似度排序全乱）

**卡住了？** 回看 [`days/day-32.md`](../days/day-32.md) 第五节「容易踩的坑」。

> **明天**：`days/day-33.md` —— 工具链完善：幂等、事务、失败降级